In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Plotting_IQR import plot_distribution, get_training_data, plot_pfn_variance_surface, plot_GP_variance_surface
from pfn_evaluate import eval_pfn
import pfns4bo
from pfns4bo.scripts.acquisition_functions import TransformerBOMethod

In [ ]:
# Load data
file_path = "AI_varied_results/run_20260805_022647/metrics.pt"
metrics = torch.load(file_path)

file_path = "AI_varied_results/run_20260805_022647/experimental_results.pt"
data = torch.load(file_path)

In [ ]:
#metrics = {
#        "pred_error": torch.zeros((n_tests, n_dims, n_methods, n_repeats, n_samples, n_fns)), # GP & PFN only
#        "total_error": torch.zeros((n_tests, n_dims, n_methods, n_repeats, 1)),
#        "EI": torch.zeros((n_tests, n_dims, n_methods, n_repeats, n_samples, 1))
#    }
pred_error = metrics["pred_error"]
total_error = metrics["total_error"]
ei = metrics["EI"]
gll = metrics["GLL"]
m_gll = metrics["MGLL"]

y_true_store = data["y_true"][0]
mu_store = data["mu"][0]
var_store = data["var"][0]
x_queried = data["x_query"][0]

ls_arr = data["lengthscales"]

In [ ]:
mu_data_GP = torch.sum(mu_store[:, 0, 0, :, :], dim=-1, keepdim=True)
mu_data_PFN = torch.sum(mu_store[:, 1, 0, :, :], dim=-1, keepdim=True)
mu_data_PFN_W = torch.sum(mu_store[:, 2, 0, :, :], dim=-1, keepdim=True)
var_data_GP = torch.sum(var_store[:, 0, 0, :, :], dim=-1, keepdim=True)
var_data_PFN = torch.sum(var_store[:, 1, 0, :, :], dim=-1, keepdim=True)
var_data_PFN_W = torch.sum(var_store[:, 2, 0, :, :], dim=-1, keepdim=True)
x_query_GP = x_queried[:, 0, 0, :, :]
x_query_PFN = x_queried[:, 1, 0, :, :]
x_query_PFN_W = x_queried[:, 2, 0, :, :]
y_true_arr = y_true_store[:, 0, 0, :, :]

In [ ]:
# metrics["pred_error"][test, k, m_idx, rep, :, :]
# 9 tests, 1 dim, 2 methods, 21 reps, 1000 samples, dim = 1 -> 9 tests, 3 methods, 1000 samples, 1 
pred_error_med = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)
total_error_med = torch.quantile(total_error[:, 0, :, :, :], 0.5, dim=-3)
ei_med = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)

pred_error_lq = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)
total_error_lq = torch.quantile(total_error[:, 0, :, :, :], 0.25, dim=-3)
ei_lq = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)

pred_error_uq = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)
total_error_uq = torch.quantile(total_error[:, 0, :, :, :], 0.75, dim=-3)
ei_uq = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)

In [ ]:
# metrics["pred_error"][test, k, m_idx, rep, :, :]
# 9 tests, 1 dim, 2 methods, 21 reps, 1000 samples, dim = 1 -> 9 tests, 3 methods, 1000 samples, 1 
gll_med = torch.quantile(torch.sum(gll[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)
mgll_med = torch.quantile(m_gll[:, 0, :, :, :], 0.5, dim=-3)

gll_lq = torch.quantile(torch.sum(gll[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)
mgll_lq = torch.quantile(m_gll[:, 0, :, :, :], 0.25, dim=-3)

gll_uq = torch.quantile(torch.sum(gll[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)
mgll_uq = torch.quantile(m_gll[:, 0, :, :, :], 0.75, dim=-3)

In [ ]:
import numpy as np

coords = np.asarray(ls_arr)  # Shape: (N, 2)

# Extract column 0 (x-axis) and column 1 (y-axis), both of shape (N,)
x = coords[:, 0]  # First coordinate of every pair
y = coords[:, 1]

# Ensure metric arrays are 2D with shape matching X and Y: (Ny, Nx)
# Example extraction assuming your metric arrays have been reshaped to (Ny, Nx, 3):
tem_GP = total_error_med[:, :, 0]
tem_PFN = total_error_med[:, :, 1]
tem_PFN_W = total_error_med[:, :, 2]

mgllm_GP = mgll_med[:, :, 0]
mgllm_PFN = mgll_med[:, :, 1]
mgllm_PFN_W = mgll_med[:, :, 2]

# Set up a 2x3 grid: Rows = Metrics, Columns = Models
fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(16, 9),
    sharex=True,
    sharey=True,
)

models = ["GP", "PFN", "PFN_W"]
error_data = [tem_GP, tem_PFN, tem_PFN_W]
ll_data = [mgllm_GP, mgllm_PFN, mgllm_PFN_W]

# Plot Row 0: Prediction Error (Median)
for col_idx, (model_name, data) in enumerate(zip(models, error_data)):
    ax = axes[0, col_idx]
    # Use pcolormesh for 2D grid plotting
    pcm = ax.pcolormesh(X, Y, data, cmap="viridis", shading="auto")
    fig.colorbar(pcm, ax=ax, label="Error")
    ax.set_title(f"Prediction Error (Median) — {model_name}")
    ax.grid(True, linestyle="--", alpha=0.3)

# Plot Row 1: Log Likelihood (Median)
for col_idx, (model_name, data) in enumerate(zip(models, ll_data)):
    ax = axes[1, col_idx]
    pcm = ax.pcolormesh(X, Y, data, cmap="plasma", shading="auto")
    fig.colorbar(pcm, ax=ax, label="Log Likelihood")
    ax.set_title(f"Log Likelihood (Median) — {model_name}")
    ax.set_xlabel("Length Scale X1")
    ax.grid(True, linestyle="--", alpha=0.3)

# Label shared Y-axis on leftmost subplots
axes[0, 0].set_ylabel("Length Scale X2")
axes[1, 0].set_ylabel("Length Scale X2")

plt.tight_layout()
plt.show()